# Imports for Shopping Chatbot

We will use the following LangChain 2026 modules:

- **ChatGroq** → to connect with the Groq LLM model.  
- **ChatPromptTemplate** → to design a structured prompt template for shopping queries.  
- **InMemoryChatMessageHistory** → to store and manage user conversation history in memory.  
- **RunnableWithMessageHistory** → to combine the prompt, LLM, and history into one runnable chain for persistent chat sessions.

In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key="YOUR_GROQ_API_KEY"
)

# Initialize the Groq LLM

We configure the **ChatGroq** model to serve as the language engine for our shopping chatbot:

- **model** → specifies the Groq model variant (`llama-3.1-8b-instant`) optimized for quick responses.  
- **api_key** → provides authentication using your Groq API key.  

This setup ensures the chatbot can generate fashion‑focused shopping recommendations while staying aligned with the defined prompt template.

In [3]:
shopping_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a shopping assistant with strong fashion sense. Only respond to queries about shopping items, fashion, or style."),
    ("human", "{question}")
])

# Create Shopping Prompt Template

We define a structured prompt that ensures the chatbot only responds to shopping and fashion-related queries:

- **system message** → sets the role of the assistant as a shopping expert with strong fashion sense.  
- **human message** → captures the user’s input dynamically through the `{question}` placeholder.  

This template restricts the chatbot’s domain to shopping items and style advice, keeping responses focused and relevant.

In [5]:
# Store user history in memory
history_store = {}

def get_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

shopping_chain = RunnableWithMessageHistory(
    shopping_prompt | llm,
    get_history,
    input_messages_key="question",
    history_messages_key="history"
)

/Users/pradipwasre/Library/Python/3.11/lib/python/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
# Store user history in memory
history_store = {}

def get_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

shopping_chain = RunnableWithMessageHistory(
    shopping_prompt | llm,
    get_history,
    input_messages_key="question",
    history_messages_key="history"
)


# Manage Conversation History

We set up memory handling so the chatbot can recall past interactions:

- **history_store** → a dictionary that keeps track of session histories.  
- **get_history(session_id)** → retrieves or initializes an `InMemoryChatMessageHistory` for each user session.  
- **RunnableWithMessageHistory** → links the shopping prompt and LLM with the history function, ensuring that user queries and responses are stored and reused.  
- **input_messages_key** → specifies the user’s input field (`question`).  
- **history_messages_key** → defines where the conversation history is maintained (`history`).  

This configuration allows the chatbot to maintain context across multiple turns, making the shopping advice more consistent and personalized.

# Test the Shopping Chatbot

We invoke the chatbot with a sample query to check its response:

- **shopping_chain.invoke** → runs the chain with the user’s input.  
- **question** → the actual user query, here asking for trendy sneakers under $100.  
- **config** → passes a `session_id` to ensure the conversation history is tied to a specific user session.  
- **print(response)** → displays the chatbot’s reply in the console.  

This step validates that the chatbot is working correctly and producing fashion‑focused shopping recommendations.

In [9]:
response = shopping_chain.invoke(
    {"question": "Suggest me trendy sneakers under rupees 1000"},
    config={"configurable": {"session_id": "user1"}}
)

print(response)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

# Build Gradio Interface

We wrap the shopping chatbot in a simple Gradio interface:

- **chat_with_bot** → function that takes user input, invokes the shopping chain, and returns the response.  
- **gr.Interface** → creates a UI with text input and output fields.  
- **title** → names the chatbot as "Shopping Fashion Chatbot".  
- **description** → explains that the assistant specializes in shopping items and fashion trends.  
- **demo.launch()** → starts the Gradio app, allowing interactive conversations with the chatbot.  

This step provides a user-friendly interface so anyone can query the chatbot about shopping and fashion directly from the browser.

In [8]:
import gradio as gr

def chat_with_bot(user_input, session_id="user1"):
    response = shopping_chain.invoke(
        {"question": user_input},
        config={"configurable": {"session_id": session_id}}
    )
    return str(response)

demo = gr.Interface(
    fn=chat_with_bot,
    inputs=["text"],
    outputs="text",
    title="Shopping Fashion Chatbot",
    description="Ask me about shopping items and fashion trends!"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
